In [1]:
import torch
import torch.nn as nn
import torch.onnx as onnx
import os
import json
import shutil
from random import randint
from ml_runner_exporter.onnx_exporter import export_onnx

In [2]:
fixtures_path = "tests/fixtures"

if os.path.exists(fixtures_path):
    shutil.rmtree(fixtures_path)
os.makedirs(fixtures_path)

In [3]:
def get_output_size() -> int:
    return randint(2, 10) * 5

In [4]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size, layer_num):
        super(SimpleLinearModel, self).__init__()
        self.layers = nn.ModuleList()
        if layer_num == 1:
            self.layers.append(nn.Linear(input_size, output_size))
        else:
            inter_output_size = get_output_size()
            self.layers.append(nn.Linear(input_size, inter_output_size))
            inter_input_size = inter_output_size
            for i in range(layer_num - 2):
                inter_output_size = get_output_size()
                self.layers.append(nn.Linear(inter_input_size, inter_output_size))
                inter_input_size = inter_output_size
            self.layers.append(nn.Linear(inter_input_size, output_size))

    def forward(self, x):
        # Pass input through the linear layer
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [5]:
# Dense layers with every activation type chained in between (relu, sigmoid,
# tanh, softmax). Linear/identity activation is intentionally excluded since
# it has no corresponding ONNX node to export from.
class ActivationModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(ActivationModel, self).__init__()
        self.linear1 = nn.Linear(input_size, 8)
        self.act1_relu = nn.ReLU()
        self.linear2 = nn.Linear(8, 12)
        self.act2_sigmoid = nn.Sigmoid()
        self.linear3 = nn.Linear(12, 10)
        self.act3_tanh = nn.Tanh()
        self.linear4 = nn.Linear(10, output_size)
        self.act4_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.linear1(x)
        x = self.act1_relu(x)
        x = self.linear2(x)
        x = self.act2_sigmoid(x)
        x = self.linear3(x)
        x = self.act3_tanh(x)
        x = self.linear4(x)
        x = self.act4_softmax(x)
        return x

In [ ]:
# Two conv layers stacked directly, no activation or flatten in between -
# isolates conv-to-conv chaining (D3 -> D3 -> D3) on its own.
class SimpleConvOnlyModel(nn.Module):
    def __init__(self):
        super(SimpleConvOnlyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=2, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x

In [ ]:
# Conv -> ReLU (D3 activation) -> Flatten, with no dense layer after -
# isolates the D3 -> Flat transition on its own.
class ConvFlattenModel(nn.Module):
    def __init__(self):
        super(ConvFlattenModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_relu = nn.ReLU()
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.act_relu(x)
        x = self.flatten(x)
        return x

In [8]:
# The full pipeline: Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear ->
# Sigmoid -> Linear -> Tanh -> Linear -> Softmax. Exercises every layer type
# and activation together in one model.
class FullConvModel(nn.Module):
    def __init__(self, output_size):
        super(FullConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, output_size)
        self.act5_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1_relu(x)
        x = self.conv2(x)
        x = self.act2_relu(x)
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.act3_sigmoid(x)
        x = self.linear2(x)
        x = self.act4_tanh(x)
        x = self.linear3(x)
        x = self.act5_softmax(x)
        return x

In [ ]:
def export_model(name: str, model: nn.Module, input_shape):
    # Avoids the "exporting a model while it is in training mode" warning;
    # doesn't change behavior here since none of these models use
    # dropout/batchnorm, but it's the right default for exported fixtures.
    model.eval()

    tmp_model_path = "temporary_model.onnx"

    # input_shape is either a flat feature count (int, for dense-only models)
    # or a (channels, height, width) tuple (for models starting with a conv
    # layer). Either way we build a batch-of-1 dummy input to trace through.
    if isinstance(input_shape, tuple):
        dummy_input_data = torch.randn(1, *input_shape)
    else:
        dummy_input_data = torch.randn(1, input_shape)

    onnx.export(model, dummy_input_data, tmp_model_path, export_params=True, opset_version=17)

    with torch.no_grad():
        output = model(dummy_input_data)

    model_output = {
        "model": export_onnx(tmp_model_path),
        # Tensor::data on the Rust side is always a flat Vec<f32> regardless
        # of TensorShape, so flatten both input and output fully here rather
        # than relying on tolist()[0] (which only drops the batch dim and
        # would leave a nested list for D3 inputs).
        "test_input": dummy_input_data.flatten().tolist(),
        "test_output": output.flatten().tolist(),
    }

    with open(os.path.join(fixtures_path, name), "w") as f:
        print(f"Exporting model: {name}")
        json.dump(model_output, f, indent=2)

In [10]:
fixtures = [
    {
        "name": "dense_simple_model.json",
        "model": SimpleLinearModel(10, 5, 1),
        "input_shape": 10,
    },
    {
        "name": "dense_long_model.json",
        "model": SimpleLinearModel(10, 5, 20),
        "input_shape": 10,
    },
    {
        "name": "dense_large_model.json",
        "model": SimpleLinearModel(100, 100, 5),
        "input_shape": 100,
    },
    {
        "name": "activation_all_types_model.json",
        "model": ActivationModel(10, 5),
        "input_shape": 10,
    },
    {
        "name": "conv_simple_model.json",
        "model": SimpleConvOnlyModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_model.json",
        "model": ConvFlattenModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_dense_activation_model.json",
        "model": FullConvModel(5),
        "input_shape": (1, 4, 4),
    },
]

In [11]:
for fixture in fixtures:
    export_model(fixture["name"], fixture["model"], fixture["input_shape"])

Exporting model: dense_simple_model.json
Exporting model: dense_long_model.json
Exporting model: dense_large_model.json
Exporting model: activation_all_types_model.json
Exporting model: conv_simple_model.json
Exporting model: conv_flatten_model.json
Exporting model: conv_flatten_dense_activation_model.json
